### ЗАДАЧА: Триаж обращений службы поддержки

Команда поддержки получает пакет строк с обращениями от разных сервисов.
Нужно обработать их так, чтобы:
- корректные обращения попали в итоговый список,
- проблемные записи не остановили весь пакет,
- по ошибкам собрался отдельный журнал,
- в конце было видно, в каких каналах остались не подтверждённые обращения,
- а также какова средняя длительность обработки по уровням приоритета.

Часть строк содержит ошибки в формате и числах,
часть использует неизвестный уровень приоритета или канал,
а часть передаёт неправильный флаг подтверждения.
        


In [ ]:
# incident_id|service|severity|duration_min|channel|acknowledged
rows = [
    'INC-100|checkout|critical|12|pager|yes',
    'INC-101|search|high|7|slack|no',
    'INC-102|billing|medium|zero|email|yes',
    'INC-103|video|critical|-3|pager|no',
    'INC-104|feed|warning|5|slack|yes',
    'INC-105|auth|low|2|sms|no',
    'INC-106|cdn|high|4|email|maybe',
    'INC-107|ml|medium|9|slack|no',
]


class IncidentProcessingError(Exception):
    pass


class IncidentFormatError(IncidentProcessingError):
    pass


class SeverityError(IncidentProcessingError):
    pass


class DurationError(IncidentProcessingError):
    pass


class ChannelError(IncidentProcessingError):
    pass


class AcknowledgedFlagError(IncidentProcessingError):
    pass


def parse_incident(row):
    # TODO: split строку по '|'
    # TODO: убрать лишние пробелы у частей через strip()
    # TODO: ожидать 6 частей: incident_id, service, severity, duration_raw, channel, acknowledged_raw
    # TODO: если частей не 6 -> raise IncidentFormatError(...)
    # TODO: duration_raw преобразовать в float
    # TODO: при ошибке преобразования использовать raise DurationError(...) from exc
    # TODO: проверить, что duration > 0
    # TODO: проверить severity в {'low', 'medium', 'high', 'critical'}
    # TODO: проверить channel в {'email', 'slack', 'pager'}
    # TODO: проверить acknowledged_raw в {'yes', 'no'}
    # TODO: превратить acknowledged_raw в bool
    # TODO: вернуть словарь с разобранными полями
    parts = [part.strip() for part in row.split('|')]
    if len(parts) != 6:
        raise IncidentFormatError(f"Ожидалось 6 полей, получено {len(parts)}: {row}")
    
    incident_id, service, severity, duration_raw, channel, acknowledged_raw = parts
    
    try:
        duration = float(duration_raw)
    except ValueError as e:
        raise DurationError(f"Некорректное значение длительности: {duration_raw}") from e
    
    if duration <= 0:
        raise DurationError(f"Длительность должна быть положительной: {duration}")
    
    valid_severities = {'low', 'medium', 'high', 'critical'}
    if severity not in valid_severities:
        raise SeverityError(f"Неизвестный уровень приоритета: {severity}")
    
    valid_channels = {'email', 'slack', 'pager'}
    if channel not in valid_channels:
        raise ChannelError(f"Неизвестный канал: {channel}")
    
    if acknowledged_raw not in {'yes', 'no'} :
        raise AcknowledgedFlagError(f"Некорректный флаг подтверждения: {acknowledged_raw}")
    acknowledged = acknowledged_raw == 'yes'
    
    return {
        'incident_id': incident_id,
        'service':  service,
        'severity': severity,
        'duration_min': duration,
        'channel': channel,
        'acknowledged': acknowledged
         }
    


def process_batch(rows):
    # TODO: создать списки incidents и errors
    # TODO: пройтись по rows циклом
    # TODO: внутри try вызвать parse_incident(row)
    # TODO: валидный incident добавить в incidents
    # TODO: IncidentProcessingError сохранить в errors как (row, error_type, message)
    # TODO: вернуть (incidents, errors)
    incidents = []
    errors = []
    
    for row in rows:
        try:
            incident = parse_incident(row)
            incidents.append(incident)
        except IncidentProcessingError as e:
            errors.append((row, type(e).__name__,str(e)))
            
    return incidents, errors


# TODO: вызвать process_batch(rows)
incidents, errors = process_batch(rows)

# TODO: вывести количество валидных инцидентов и количество ошибок
print(f"Количество валидных инцидентов: {len(incidents)}")
print(f"Количество ошибок: {len(errors)}")

# TODO: собрать error_counts: dict[str, int]
error_counts = {}
for _, error_type, _ in errors:
    error_counts[error_type] = error_counts.get(error_type, 0) + 1
print("Статистика по ошибкам:")
for error_type, count in error_counts.items():
    print(f" {error_type}: {count}")
print()

# TODO: собрать unacked_by_channel: dict[str, list[str]] только для acknowledged == False
unacked_by_channel = {}
for incident in incidents:
    if not incident['acknowledged']:
        channel = incident['channel']
        if channel not in unacked_by_channel:
            unacked_by_channel[channel] = []
        unacked_by_channel[channel].append(incident['incident_id'])
        
print("Не подтвержденные обращения по каналам:")
if unacked_by_channel:
    for channel, incidents_list in unacked_by_channel.items():
        print(f" Канал '{channel}': {incidents_list}")
else:
    print(" Нет не подтвержденных обращений")
print()
        
# TODO: собрать average_duration_by_severity только по валидным строкам
average_duration_by_severity = {}
severity_durations = {}

for incident in incidents:
    severity = incident['severity']
    duration = incident['duration_min']
    if severity not in severity_durations:
        severity_durations[severity] = []
    severity_durations[severity].append(duration)
    
for severity, durations in severity_durations.items():
    average_duration_by_severity[severity] = sum(durations) / len(durations)
    
print("Средняя длительность обработки по уровням приоритета:")
for severity in sorted(average_duration_by_severity.keys()):
    avg_duration = average_duration_by_severity[severity]
    print(f" {severity}: {avg_duration:.2f} мин")
print()

# TODO: найти longest_incident среди валидных инцидентов по duration_min
if incidents:
    longest_incident = max(incidents, key=lambda x: x['duration_min'])
    print("Самый длительный интидент")
    print(f" ID: {longest_incident['incident_id']}")
    print(f" Сервис: {longest_incident['service']}")
    print(f" Приоритет: {longest_incident['severity']}")
    print(f" Длительность: {longest_incident['duration_min']} мин")
    print(f" Канал: {longest_incident['channel']}")
    print(f" Подтвержден: {'Да' if longest_incident['acknowledged'] else 'Нет'}")
else:
    print("Валидных инцидентов нет, самый длительный инцидент не найден")
    
# TODO: красиво вывести получившиеся структуры
print("Подробный журнал ошибок:")
for row, error_type, message in errors:
    print(f" Строка: '{row}'")
    print(f"    Ошибка: {error_type}")
    print(f"    Сообщение: {message}")